## Concept primer: pandas cleaning toolkit

A DataFrame `df` is a table. `df["col"]` is one column (a Series). You clean a column by **reassigning it**: `df["col"] = <cleaned version>`.

| Need | Code |
|---|---|
| look at it | `df.head()`, `df.info()`, `df.isna().sum()`, `df["col"].value_counts(dropna=False)` |
| remove exact duplicate rows | `df = df.drop_duplicates()` |
| keep rows matching a condition | `df = df[df["col"] > 0]` |
| string ops on a whole column | `.str.strip()`, `.str.lower()`, `.str.title()`, `.str.replace("a", "b", regex=False)` (chain them) |
| text → number | `pd.to_numeric(series)` (add `errors="coerce"` to turn junk into NaN) |
| text → date | `pd.to_datetime(series, errors="coerce")` (junk → `NaT`, the date version of NaN) |
| fill missing | `series.fillna(value)`, e.g. `series.fillna(series.median())` |
| recode values | `series.map({"old": new, ...})` (anything not in the dict becomes NaN) |

**When to use median not mean for filling:** skewed data with outliers (income, transaction amounts). One £950k salary drags the mean up; the median barely moves.

**Tiny example:**
```python
s = pd.Series([" Leeds", "LEEDS", "leeds "])
s.str.strip().str.title()      # -> Leeds, Leeds, Leeds
```

## Problem: clean a messy loan-applications file

`data/loans_messy.csv` has one row per loan application: `application_id, application_date, region, employment, income, loan_amount, defaulted`.

It's messy. **Do the steps in order** (later tests depend on earlier ones). Each step has its own test cell. Run it after the step.

**Target: 25 minutes.** Look things up in the primer table above, that's allowed.

In [1]:
import pandas as pd
df = pd.read_csv("../data/loans_messy.csv")

# Step 0: inspect. Run these, then write in the cell below what's messy (3-5 bullet points).
print(df.shape)
df.info()
print(df.isna().sum())
df.head(15)


(495, 7)
<class 'pandas.DataFrame'>
RangeIndex: 495 entries, 0 to 494
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   application_id    495 non-null    str    
 1   application_date  495 non-null    str    
 2   region            495 non-null    str    
 3   employment        469 non-null    str    
 4   income            457 non-null    str    
 5   loan_amount       495 non-null    float64
 6   defaulted         495 non-null    str    
dtypes: float64(1), str(6)
memory usage: 27.2 KB
application_id       0
application_date     0
region               0
employment          26
income              38
loan_amount          0
defaulted            0
dtype: int64


,application_id,application_date,region,employment,income,loan_amount,defaulted
0,A10374,2025-12-26,Leeds,Self-Employed,24600,15800.0,1
1,A10267,2025-07-24,Leeds,Unemployed,"£34,600",16800.0,N
2,A10372,2025-05-31,London,employed,56400,16400.0,yes
3,A10161,2025-02-15,Glasgow,employed,38700,14800.0,1
4,A10327,2025-01-23,London,UNEMPLOYED,69100,2100.0,1
5,A10413,2025-10-06,Glasgow,EMPLOYED,"£68,900",5200.0,0
6,A10469,2025-11-12,London,self-employed,"£106,500",20200.0,N
7,A10410,2025-05-24,Manchester,employed,25900,13500.0,1
8,A10398,2025-08-10,London,NaN,20500,29700.0,1
9,A10390,2025-03-07,Manchester,employed,42700,26400.0,N


_Step 0 - what's messy in this data:_

✅ **Model answer**

- duplicate rows (500 rows, 20 exact duplicates)
- impossible loan amounts (0 or negative)
- `region` with mixed case and stray spaces (`' Leeds'`, `'LEEDS '`)
- `employment` with spelling/case variants and missing values
- `income` stored as text with `£` and commas, plus missing values
- `defaulted` mixing `1/0`, `Y/N`, `yes/no`
- `application_date` as text, with `'not provided'` in some rows

**Step 1:** remove exact duplicate rows.

In [2]:
df = df.drop_duplicates()

In [3]:
assert len(df) == 480, len(df)
print('step 1 ok')


step 1 ok


**Step 2:** a loan amount of 0 or less is impossible. Keep only rows with `loan_amount > 0`.

In [4]:
df = df[df["loan_amount"] > 0]

In [5]:
assert len(df) == 474, len(df)
assert (df['loan_amount'] > 0).all()
print('step 2 ok')


step 2 ok


**Step 3:** `region` has inconsistent case and stray spaces (`' Leeds'`, `'LEEDS '`). Standardise to `Leeds`, `London`, `Manchester`, `Glasgow`.

In [6]:
df["region"] = df["region"].str.strip().str.title()

In [7]:
assert sorted(df['region'].unique()) == ['Glasgow', 'Leeds', 'London', 'Manchester'], df['region'].unique()
print('step 3 ok')


step 3 ok


**Step 4:** `employment` has variants (`'Self Employed'`, `'self employed'`, `'EMPLOYED'` ...) and missing values.
Standardise to exactly `employed`, `self-employed`, `unemployed`, and fill missing with `unknown`.

In [8]:
df["employment"] = (df["employment"].str.strip().str.lower()
                    .str.replace(" ", "-", regex=False)
                    .fillna("unknown"))

In [9]:
assert sorted(df['employment'].unique()) == ['employed', 'self-employed', 'unemployed', 'unknown'], df['employment'].unique()
print('step 4 ok')


step 4 ok


**Step 5:** `income` is text: some rows are `'£34,600'`, some `'34600'`, some missing.
Convert it to a **float** column, then fill missing values with the **median** income.

In [10]:
df["income"] = pd.to_numeric(df["income"].str.replace("£", "", regex=False)
                                        .str.replace(",", "", regex=False))
df["income"] = df["income"].fillna(df["income"].median())

In [11]:
assert pd.api.types.is_float_dtype(df['income']), df['income'].dtype
assert df['income'].isna().sum() == 0
assert df['income'].median() == 36500.0, df['income'].median()
print('step 5 ok')


step 5 ok


**Step 6:** `defaulted` is a mix of `'1'`, `'0'`, `'Y'`, `'N'`, `'yes'`, `'no'`. Convert to an **integer** column of 0/1.
(Tip: check `df['defaulted'].value_counts()` first.)

In [12]:
print(df["defaulted"].value_counts())
df["defaulted"] = (df["defaulted"].astype(str).str.strip().str.lower()
                   .map({"1": 1, "y": 1, "yes": 1, "0": 0, "n": 0, "no": 0})
                   .astype(int))

defaulted
0      261
1       95
N       47
no      41
Y       16
yes     14
Name: count, dtype: int64


In [13]:
assert pd.api.types.is_integer_dtype(df['defaulted']), df['defaulted'].dtype
assert set(df['defaulted'].unique()) == {0, 1}
assert df['defaulted'].sum() == 125, df['defaulted'].sum()
print('step 6 ok')


step 6 ok


**Step 7:** convert `application_date` to datetime. Some rows say `'not provided'`: they should become `NaT`, not crash.

In [14]:
df["application_date"] = pd.to_datetime(df["application_date"], errors="coerce")   # 'not provided' -> NaT

In [15]:
assert pd.api.types.is_datetime64_any_dtype(df['application_date']), df['application_date'].dtype
assert df['application_date'].isna().sum() == 11
print('step 7 ok - data is clean')


step 7 ok - data is clean


## Explain it (assumptions: graded)

Answer in 1 line each:
1. Why fill income with the **median** and not the mean? (Look at `df['income'].describe()`.)
2. In step 2 you **dropped** bad rows. In step 7 you **kept** rows with missing dates. Why is that a reasonable difference?
3. There are a few incomes of £950,000. Would you remove them? What would you check first?

✅ **Model answer**

1. Income is **right-skewed** (a few incomes of £950,000 pull the mean well above the median), so the median is a more typical value to fill with and isn't dragged by outliers.
2. A loan amount ≤ 0 is **impossible**, so the whole row is untrustworthy; a missing date is just **unknown**, and the rest of the row (income, default) is still valid and useful, so keep it as `NaT`.
3. Not automatically. First check whether they're **plausible** (job title, loan size, other fields) or a **data-entry error** (e.g. an extra zero, or the wrong unit). Real high earners should stay; for modelling you might cap them or use a log scale rather than delete them.